# Combinations with Lensing

In this notebook, we will look at the Lensing likelihood and look at the effect that lensing has on CMB constraints.

In [1]:
import jax
from jax import numpy as jnp
from jax import scipy as jsc
from jax import random as jrd

import numpy as np
import matplotlib.pyplot as plt

import camb

import sys
sys.path.insert(0, "../")

import mflike_jax
import mflike_jax.util

import pprint

try:
    from tqdm.notebook import tqdm
except:
    tqdm = lambda x: x

/home/ojima/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
# Please see notebook 3 for installation instructions on Cosmopower.
import sys

sys.path.insert(0, "cosmopower")

import cosmopower as cp
from cosmopower_jax.cosmopower_jax import CosmoPowerJAX

I0000 00:00:1790108498.159981   20960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790108499.977223   20960 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
parser = cp.YAMLParser("jense_2024_emulators/jense_2023_cmb_camb_lcdm.yaml", root_dir="jense_2024_emulators")

emulators = mflike_jax.util.emulators_to_jax(parser, desired=["tt", "te", "ee", "pp"])

cmb_tt = emulators["tt"]
cmb_te = emulators["te"]
cmb_ee = emulators["ee"]
cmb_pp = emulators["pp"]

Failed to restore network derived: Failed to restore network from jense_2024_emulators/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_derived:  does not exist..


E0000 00:00:1790108501.339839   20960 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1790108501.340241   21034 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1790108501.362168   20960 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Failed to restore network Hubble: Failed to restore network from jense_2024_emulators/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_Hubble:  does not exist..
Failed to restore network angular_diameter_distance: Failed to restore network from jense_2024_emulators/jense_2023_camb_lcdm/networks/jense_2023_camb_lcdm_angular_diameter_distance:  does not exist..


In [4]:
like = mflike_jax.Lensing_jax("so_example_lensing.yaml")

In [5]:
print(cmb_pp.parameters)
theta = jnp.array([0.022, 0.117, 3.044, 0.96, 0.67])

dlpp = cmb_pp.predict(theta)

chi2 = like.chisquare(dlpp)

print(f"chi square = {chi2:.3f}")

['ombh2', 'omch2', 'logA', 'ns', 'h']
chi square = 64.425


In [6]:
# The lensing likelihood needs to be corrected for the cosmology.
# This has been implemented similar to the primary CMB foregrounds.
lensing_corrections = mflike_jax.LensingCorrections("so_example_lensing_corrections.yaml")

In [7]:
theta_cmb = jnp.array([0.022, 0.117, 3.044, 0.06, 0.96, 0.67])

dltt = cmb_tt.predict(theta_cmb)
dlte = cmb_te.predict(theta_cmb)
dlee = cmb_ee.predict(theta_cmb)
# Cosmopower-jax doesn't like BB emulators :(
# We'll just take zero BB modes for now.
dlbb = np.zeros_like(dltt)

corr = lensing_corrections.get_corrections(dltt, dlte, dlee, dlbb, dlpp)

chi2 = like.chisquare(dlpp, corr)
print(f"chi square = {chi2:.3f}")

chi square = 55.397
